In [1]:
from crewai import Agent, Task, Crew, Process, LLM
import PyPDF2

def extract_pdf_text(pdf_path):
    with open(pdf_path, 'rb') as f:
        reader = PyPDF2.PdfReader(f)
        text = ""
        for page in reader.pages:
            text += page.extract_text()
    return text

In [2]:
llm = LLM(
    model="ollama/llama3.1:8b",
    base_url="http://localhost:11434",
)

# Extract PDF content
pdf_path = '/home/ilham/Documents/python/crewai-vs-langgraph/doc/220920 pseudocode dan golang dasar.pdf'
pdf_content = extract_pdf_text(pdf_path)

In [ ]:
# 3. Definisi Rubrik Scoring (Sesuai permintaan Anda)
SCORING_RUBRIC = """
SISTEM PENILAIAN (Maksimal 20 Poin per Soal):

1. Kesalahan Kritis:
   - Logika inti sepenuhnya salah, menghasilkan output tidak relevan: -10 Poin
   - Program tidak menyelesaikan masalah sama sekali (ada usaha): -18 Poin
   - Lembar jawaban kosong: -20 Poin

2. Kesalahan Logika Mayor:
   - Gagal menangani salah satu kondisi utama: -8 Poin
   - Perhitungan matematis utama tidak akurat: -8 Poin
   - Variabel tidak tertulis pada kamus: -5 Poin

3. Kesalahan Logika Minor & Struktur:
   - Gagal menangani kasus khusus (edge case): -5 Poin
   - Tidak menggunakan tipe bentukan (jika diwajibkan): -5 Poin
   - Alur program tidak efisien/berbelit: -3 Poin

4. Kesalahan Kelengkapan:
   - Tipe data variabel tidak sesuai: -2 Poin
   - Penulisan variabel berbeda (typo) antara kamus & program: -1 Poin
   - Format output tidak sesuai: -1 Poin
"""

# 4. Mendefinisikan Agent
grader_agent = Agent(
    role="Senior Algorithm & Logic Grader",
    goal="Menilai jawaban siswa secara ketat berdasarkan soal dan rubrik pengurangan poin.",
    backstory=(
        "Anda adalah asisten dosen informatika yang sangat ketat.\n\n"
        "ATURAN WAJIB:\n"
        "- Anda HARUS mencocokkan KONDISI LOGIKA dengan NARASI SOAL.\n"
        "- Jika arah kondisi if/else terbalik, itu adalah KESALAHAN LOGIKA MAYOR.\n"
        "- Anda DILARANG membuat sistem penilaian sendiri.\n"
        "- Anda HANYA boleh menggunakan RUBRIK yang diberikan.\n\n"
        "Panduan penulisan pseudocode:\n"
        f"{pdf_content}"
    ),
    llm=llm,
    verbose=True
)

reviewer_agent = Agent(
    role="Academic Logic Reviewer",
    goal="Memverifikasi ulang logika dan perhitungan skor. Berhak membatalkan penilaian salah.",
    backstory=(
        "Anda adalah dosen pengampu.\n\n"
        "TUGAS WAJIB:\n"
        "- Periksa ulang logika if/else dibandingkan SOAL.\n"
        "- Jika grader salah menilai logika, ANDA WAJIB memperbaiki skor.\n"
        "- Pastikan skor = 20 - total_pengurangan.\n"
        "- Jangan bersikap permisif."
    ),
    llm=llm,
    verbose=True
)

grading_task = Task(
    description=(
        "Nilai jawaban siswa berdasarkan soal.\n\n"
        "SOAL:\n{soal}\n\n"
        "JAWABAN SISWA:\n{jawaban_siswa}\n\n"
        "ATURAN KETAT:\n"
        "1. Cocokkan LOGIKA if/else dengan narasi soal.\n"
        "2. Jika kondisi TERBALIK, beri -8 poin.\n"
        "3. Jika logika benar dan lengkap, skor 20.\n"
        "4. Gunakan rubrik berikut SAJA:\n{rubrik}\n\n"
        "FORMAT OUTPUT WAJIB:\n"
        "- Temuan Kesalahan:\n"
        "- Kategori Rubrik:\n"
        "- Total Pengurangan:\n"
        "- Skor Sementara:"
    ),
    expected_output="Laporan analisis terstruktur.",
    agent=grader_agent
)

review_task = Task(
    description=(
        "Review hasil penilaian.\n"
        "VERIFIKASI ULANG:\n"
        "- Logika if/else\n"
        "- Total pengurangan\n"
        "- Skor akhir\n\n"
        "Jika ada kesalahan penilaian, PERBAIKI."
    ),
    expected_output=(
        "Tabel Markdown final berisi:\n"
        "| Temuan | Pengurangan | Skor Akhir |"
    ),
    agent=reviewer_agent
)

grading_crew = Crew(
    agents=[grader_agent, reviewer_agent],
    tasks=[grading_task, review_task],
    process=Process.sequential
)


# 7. Eksekusi (Contoh Input)
input_data = {
    'soal': "Buatlah pseudocode untuk menghitung nilai akhir mahasiswa. Jika nilai > 75 lulus, jika tidak gagal. Variabel harus ada di kamus.",
    'jawaban_siswa': """
        Program HitungLulus
        kamus
            n : integer
        algoritma
            input(n)
            if n < 75 then
                print("Lulus")
            else
                print("Gagal")
        endprogram
    """,
    'rubrik': SCORING_RUBRIC
}

print("### MEMULAI PROSES PENILAIAN ###")
result = grading_crew.kickoff(inputs=input_data)

print("\n\n########################")
print("## HASIL PENILAIAN AKHIR ##")
print("########################\n")
print(result)

### MEMULAI PROSES PENILAIAN ###


╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Senior Algorithm & Logic Grader                                                                         │
│                                                                                                                 │
│  Task: Nilai jawaban siswa berdasarkan soal.                                                                    │
│                                                                                                                 │
│  SOAL:                                                                                                          │
│  Buatlah pseudocode untuk menghitung nilai akhir mahasiswa. Jika nilai > 75 lulus, jika tidak gagal. Variabel   │
│  harus ada di kamus.                                                                                            │
│                                                                                                                 │
│  JAWABAN SISWA:                                                                                                 │
│                                                                                                                 │
│          Program HitungLulus                                                                                    │
│          kamus                                                                                                  │
│              n : integer                                                                                        │
│          algoritma                                                                                              │
│              input(n)                                                                                           │
│              if n < 75 then                                                                                     │
│                  print("Lulus")                                                                                 │
│              else                                                                                               │
│                  print("Gagal")                                                                                 │
│          endprogram                                                                                             │
│                                                                                                                 │
│                                                                                                                 │
│  ATURAN KETAT:                                                                                                  │
│  1. Cocokkan LOGIKA if/else dengan narasi soal.                                                                 │
│  2. Jika kondisi TERBALIK, beri -8 poin.                                                                        │
│  3. Jika logika benar dan lengkap, skor 20.                                                                     │
│  4. Gunakan rubrik berikut SAJA:                                                                                │
│                                                                                                                 │
│  SISTEM PENILAIAN (Maksimal 20 Poin per Soal):                                                                  │
│                                                                                                                 │
│  1. Kesalahan Kritis:                                                                                           │
│     - Logika inti sepenuhnya salah, menghasilkan output tidak relevan: -10 Poin                                 │
│     - Program tidak menyelesaikan masalah sama sekali (

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Senior Algorithm & Logic Grader                                                                         │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  ### Temuan Kesalahan:                                                                                          │
│  - Nilai variabel "n" tidak diubah dalam kondisi if/else.                                                       │
│  - Logika inti sepenuhnya salah, menghasilkan output yang tidak relevan.                                        │
│                                                                                                                 │
│  ### Kategori Rubrik:                                                                                           │
│  - Kesalahan Logika Mayor: -8 Poin                                                                              │
│                                                                                                                 │
│  ### Total Pengurangan: 8 Poin                                                                                  │
│                                                                                                                 │
│  ### Skor Sementara: 12 Poin (Maksimal 20 Poin)                                                                 │
│                                                                                                                 │
│  Penguraian skor adalah sebagai berikut:                                                                        │
│                                                                                                                 │
│  * Kesalahan Logika Mayor: -8 Poin karena nilai variabel "n" tidak diubah dalam kondisi if/else.                │
│  * Logika inti sepenuhnya salah, menghasilkan output yang tidak relevan.                                        │
│                                                                                                                 │
│  Catatan: Skor sementara dapat diperbarui jika ada kesalahan lain atau penemuan baru.                           │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Academic Logic Reviewer                                                                                 │
│                                                                                                                 │
│  Task: Review hasil penilaian.                                                                                  │
│  VERIFIKASI ULANG:                                                                                              │
│  - Logika if/else                                                                                               │
│  - Total pengurangan                                                                                            │
│  - Skor akhir                                                                                                   │
│                                                                                                                 │
│  Jika ada kesalahan penilaian, PERBAIKI.                                                                        │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Academic Logic Reviewer                                                                                 │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  | Temuan | Pengurangan | Skor Akhir |                                                                          │
│  | --- | --- | --- |                                                                                            │
│  | Nilai variabel "n" tidak diubah dalam kondisi if/else (Kesalahan Logika Mayor) | -8 Poin | 12 Poin           │
│  (Maksimal 20 Poin) - 8 Poin = 4 Poin |                                                                         │
│  | Logika inti sepenuhnya salah, menghasilkan output yang tidak relevan | -0 Poin (dalam kategori ini           │
│  kesalahan sudah tercakup dalam Kesalahan Logika Mayor) |  |                                                    │
│  | Total Pengurangan | 8 Poin |  |                                                                              │
│  | Skor Akhir | 4 Poin |  |                                                                                     │
│                                                                                                                 │
│  Catatan: Saya telah memperbarui skor dengan menghilangkan kesalahan yang sudah tercakup dalam Kesalahan        │
│  Logika Mayor.                                                                                                  │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯



########################
## HASIL PENILAIAN AKHIR ##
########################

| Temuan | Pengurangan | Skor Akhir |
| --- | --- | --- |
| Nilai variabel "n" tidak diubah dalam kondisi if/else (Kesalahan Logika Mayor) | -8 Poin | 12 Poin (Maksimal 20 Poin) - 8 Poin = 4 Poin |
| Logika inti sepenuhnya salah, menghasilkan output yang tidak relevan | -0 Poin (dalam kategori ini kesalahan sudah tercakup dalam Kesalahan Logika Mayor) |  |
| Total Pengurangan | 8 Poin |  |
| Skor Akhir | 4 Poin |  |

Catatan: Saya telah memperbarui skor dengan menghilangkan kesalahan yang sudah tercakup dalam Kesalahan Logika Mayor.
